In [54]:
import os
import shutil
import csv
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from skimage import io, transform

import torch
from torch import optim
from torch import nn
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm

import torchvision

import torch.nn.functional as F
import torchvision.datasets as datasets
import torchvision.transforms as transforms

import torchmetrics
import kagglehub

from statsmodels.stats.outliers_influence import variance_inflation_factor

In [21]:
path1 = kagglehub.dataset_download("shree0910/online-vs-in-store-shopping-behaviour-dataset")
path2 = kagglehub.dataset_download("vishardmehta/smartphone-battery-health-prediction-dataset")
path3 = kagglehub.dataset_download("foolishboi/blood-transfusion")
path4 = kagglehub.dataset_download("yasserh/wine-quality-dataset")

In [157]:
shopping_data = pd.read_csv(f"{path1}/{os.listdir(path1)[0]}")
smartphone_data = pd.read_csv(f"{path2}/{os.listdir(path2)[0]}")
smartphone_features = pd.read_csv(f"{path2}/{os.listdir(path2)[1]}")
blood_data = pd.read_csv(f"{path3}/{os.listdir(path3)[0]}")
wine_data = pd.read_csv(f"{path4}/{os.listdir(path4)[0]}")

In [158]:
def cast_int(df, column):
    dict = {}
    i = 0
    for label in df[column].unique():
        dict[label] = i
        i += 1
    
    return df[column].map(dict)

In [159]:
shopping_data["shopping_preference"] = cast_int(shopping_data, "shopping_preference")
shopping_data["city_tier"] = cast_int(shopping_data, "city_tier")
shopping_data["gender"] = cast_int(shopping_data, "gender")

In [160]:
shopping_data["shopping_preference"] = shopping_data["shopping_preference"].map({0: 0, 1: 1, 2:1})

In [161]:
def vifify(df, column):
    X = df.drop(column, axis=1)

    vif_data = pd.DataFrame()
    vif_data["feature"] = X.columns

    vif_data["VIF"] = [variance_inflation_factor(X.values, i)
                            for i in range(len(X.columns))]
    
    vif_data.sort_values(by="VIF", axis=0, ascending=False, inplace=True)
    
    return vif_data, X

In [162]:
def del_coll(df, col, threshold=5):
    column = col
    target = df[col]
    new_shop = df
    while True:
        vif, new_shop = vifify(new_shop, column)

        if vif.iloc[0,1] > threshold:
            column = vif.iloc[0,0]
        
        else:
            print(vif)
            break

    new_shop[col] = target
    
    return new_shop

In [163]:
shopping_data = del_coll(shopping_data, "shopping_preference")

                        feature       VIF
11            avg_delivery_days  4.768549
2            social_media_hours  4.728235
17          brand_loyalty_score  4.548954
0                monthly_income  4.505733
14  product_availability_online  4.479064
19          time_pressure_level  4.426047
4              tech_savvy_score  4.419042
15         impulse_buying_score  4.417325
16        need_touch_feel_score  4.406046
12     delivery_fee_sensitivity  4.405095
9          discount_sensitivity  4.402583
3    online_payment_trust_score  4.396035
18      environmental_awareness  4.393591
13       free_return_importance  4.371875
1        smartphone_usage_years  4.355952
8               avg_store_spend  3.879587
7              avg_online_spend  3.836766
5         monthly_online_orders  3.779715
6          monthly_store_visits  3.609372
10             return_frequency  3.281533
20                       gender  2.467068
21                    city_tier  2.450476


In [164]:
smartphone_data = smartphone_features.merge(smartphone_data, on="Device_ID").drop("current_battery_health_percent", axis=1)

In [165]:
smartphone_data["target_action"] = smartphone_data["recommended_action"].map({"Change Phone":1, "Replace Battery":1, "Keep Using":0})
smartphone_data = smartphone_data.drop(["recommended_action", "Device_ID"], axis=1)
smartphone_data["background_app_usage_level"] = cast_int(smartphone_data, "background_app_usage_level")
smartphone_data["signal_strength_avg"] = cast_int(smartphone_data, "signal_strength_avg")
smartphone_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 15 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   device_age_months                 5000 non-null   int64  
 1   battery_capacity_mah              5000 non-null   int64  
 2   avg_screen_on_hours_per_day       5000 non-null   float64
 3   avg_charging_cycles_per_week      5000 non-null   float64
 4   avg_battery_temp_celsius          5000 non-null   float64
 5   fast_charging_usage_percent       5000 non-null   float64
 6   overnight_charging_freq_per_week  5000 non-null   int64  
 7   gaming_hours_per_week             5000 non-null   float64
 8   video_streaming_hours_per_week    5000 non-null   float64
 9   background_app_usage_level        5000 non-null   int64  
 10  signal_strength_avg               5000 non-null   int64  
 11  charging_habit_score              5000 non-null   int64  
 12  usage_

In [167]:
smartphone_data = del_coll(smartphone_data, "target_action", threshold=10)

                            feature       VIF
1      avg_charging_cycles_per_week  6.472615
2       fast_charging_usage_percent  5.237983
7               signal_strength_avg  4.501889
5    video_streaming_hours_per_week  4.358368
0                 device_age_months  3.542441
3  overnight_charging_freq_per_week  3.020644
4             gaming_hours_per_week  2.825954
6        background_app_usage_level  2.350005


In [171]:
smartphone_data[smartphone_data.duplicated()]

,device_age_months,avg_charging_cycles_per_week,fast_charging_usage_percent,overnight_charging_freq_per_week,gaming_hours_per_week,video_streaming_hours_per_week,background_app_usage_level,signal_strength_avg,target_action


In [ ]:
# blood_data.drop_duplicates(inplace=True, ignore_index=True)
# blood_data.info()
# blood_data["Class"] = blood_data["Class"].map({1:0, 2:1})

In [179]:
blood_data_new = del_coll(blood_data, "Class", 10)

  feature       VIF
2      V4  5.550033
1      V3  3.558410
0      V1  2.201671


c:\Users\tomas\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\stats\outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)


In [ ]:
wine_data.info()
# wine_data = wine_data.drop('Id', axis=1)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1143 entries, 0 to 1142
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed acidity         1143 non-null   float64
 1   volatile acidity      1143 non-null   float64
 2   citric acid           1143 non-null   float64
 3   residual sugar        1143 non-null   float64
 4   chlorides             1143 non-null   float64
 5   free sulfur dioxide   1143 non-null   float64
 6   total sulfur dioxide  1143 non-null   float64
 7   density               1143 non-null   float64
 8   pH                    1143 non-null   float64
 9   sulphates             1143 non-null   float64
 10  alcohol               1143 non-null   float64
 11  quality               1143 non-null   int64  
dtypes: float64(11), int64(1)
memory usage: 107.3 KB


In [187]:
wine_data["quality"].unique()

array([5, 6, 7, 4, 8, 3])

In [188]:
def cast_wine(df, column):
    dict = {}
    for label in df[column].unique():
        if label > 5:
            dict[label] = 1
        else:
            dict[label] = 0
    
    return df[column].map(dict)

wine_data["quality"] = cast_wine(wine_data, "quality")

In [189]:
wine_data = del_coll(wine_data, "quality", 10)

                feature       VIF
4   free sulfur dioxide  5.642393
0      volatile acidity  5.513306
5  total sulfur dioxide  5.490251
3             chlorides  4.906418
2        residual sugar  4.730015
1           citric acid  3.176401


### Assumed binary values for each 
- Shopping: 0 -- stationary,   1 -- online
- Smartphones: 0 -- no action,  1 -- replace battery or phone
- Blood: 0 -- did not donate,  1 -- donated
- Wine: 0 -- poor quality,  1 -- good quality